# RetailMart Lakehouse

## Notebook: 01_Load_Customers

### Layer
Bronze Layer

### Objective
This notebook ingests the raw customer dataset into the Bronze layer of the RetailMart Lakehouse.

The notebook performs the following tasks:

1. Reads raw customer data from the Raw Volume.
2. Performs initial data profiling.
3. Validates the dataset structure.
4. Adds audit metadata.
5. Loads data into a Delta Bronze table.
6. Performs post-load validation.

### Source

Volume:
`/Volumes/dbacademy/default/raw/raw_customers_dataset.csv`

### Target

`retailmart.bronze.customers`

### Version

1.0

In [0]:
%run ../00_Setup/01_Config

In [0]:
%run ../Utils/Common_Utils

In [0]:
# Imports

from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import uuid

In [0]:
# Pipeline Configuration

SOURCE_FILE = RAW_CUSTOMERS          
TARGET_TABLE = BRONZE_CUSTOMERS      
PIPELINE_NAME = "Bronze_Customers"
LAYER = "Bronze"

In [0]:
# Pipeline Metadata - Generate run information 

RUN_ID = generate_run_id()

START_TIME = start_pipeline()

print(f"Pipeline  : {PIPELINE_NAME}")
print(f"Run ID    : {RUN_ID}")
print(f"Start Time: {START_TIME}")

Pipeline Started : 2026-07-15 07:36:59.737479
Pipeline  : Bronze_Customers
Run ID    : c5cd7a7e-f6cb-4a1a-b298-b80c72a8ca3b
Start Time: 2026-07-15 07:36:59.737479


In [0]:
customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_unique_id", StringType(), True),
    StructField("customer_zip_code_prefix", IntegerType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True)
])

In [0]:
customers_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(SOURCE_FILE)
)

In [0]:
# Cache the dataframe since we'll use it multiple times
# (count, display, write) — avoids re-reading CSV each time
# customers_df.cache()

total_rows = customers_df.count()
total_columns = len(customers_df.columns)

print("DATA PROFILE")
print(f"Rows    : {total_rows}")
print(f"Columns : {total_columns}")
print("\nSchema")
customers_df.printSchema()

DATA PROFILE
Rows    : 15000
Columns : 5

Schema
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [0]:
# Expected Schema (reference)
EXPECTED_SCHEMA = {
    "customer_id": "string",
    "customer_unique_id": "string",
    "customer_zip_code_prefix": "int",
    "customer_city": "string",
    "customer_state": "string"
}

In [0]:
schema_status = validate_schema(
    customers_df,
    EXPECTED_SCHEMA
)

Schema Validation Passed


In [0]:
display(customers_df.limit(10))

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
CUST_000001,UNIQ_000001,13278,salvador,BA
CUST_000002,UNIQ_000002,42098,porto alegre,RS
CUST_000003,UNIQ_000003,28289,recife,PE
CUST_000004,UNIQ_000004,98696,salvador,BA
CUST_000005,UNIQ_000005,21395,maceio,AL
CUST_000006,UNIQ_000006,65302,cuiaba,MT
CUST_000007,UNIQ_000007,13905,rio de janeiro,RJ
CUST_000008,UNIQ_000008,38657,belo horizonte,MG
CUST_000009,UNIQ_000009,76237,recife,PE
CUST_000010,UNIQ_000010,13478,porto velho,RO


In [0]:
pk_status = validate_primary_key(
    customers_df,
    "customer_id"
)

Total Rows : 15000
Distinct Count : 15000
Primary Key Validation Passed — (customer_id)


duplicate_rows = total_rows - customers_df.dropDuplicates().count()
print(f"Duplicate Rows : {duplicate_rows}")

In [0]:
duplicate_rows = duplicate_summary(customers_df,total_rows)

Duplicate Rows : 0


In [0]:
null_summary(customers_df)

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,0,0,0,0


In [0]:
customers_df.describe().show()

+-------+-----------+------------------+------------------------+-------------+--------------+
|summary|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-------+-----------+------------------+------------------------+-------------+--------------+
|  count|      15000|             15000|                   15000|        15000|         15000|
|   mean|       NULL|              NULL|              54722.2996|         NULL|          NULL|
| stddev|       NULL|              NULL|      26037.492029835856|         NULL|          NULL|
|    min|CUST_000001|       UNIQ_000001|                   10004|      aracaju|            AL|
|    max|CUST_015000|       UNIQ_015000|                   99996|     teresina|            SP|
+-------+-----------+------------------+------------------------+-------------+--------------+



In [0]:
dataset_profile(customers_df,RAW_CUSTOMERS)

/Volumes/dbacademy/default/raw/raw_customers_dataset.csv
Rows    : 15000
Columns : 5
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
CUST_000001,UNIQ_000001,13278,salvador,BA
CUST_000002,UNIQ_000002,42098,porto alegre,RS
CUST_000003,UNIQ_000003,28289,recife,PE
CUST_000004,UNIQ_000004,98696,salvador,BA
CUST_000005,UNIQ_000005,21395,maceio,AL
CUST_000006,UNIQ_000006,65302,cuiaba,MT
CUST_000007,UNIQ_000007,13905,rio de janeiro,RJ
CUST_000008,UNIQ_000008,38657,belo horizonte,MG
CUST_000009,UNIQ_000009,76237,recife,PE
CUST_000010,UNIQ_000010,13478,porto velho,RO


from pyspark.sql.functions import current_timestamp, current_date, lit

customers_df = (
    customers_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("ingestion_date", current_date())
    .withColumn("pipeline_name", lit("Bronze_Customers"))
    .withColumn("run_id", lit(RUN_ID))
)

In [0]:
customers_df = add_audit_columns(customers_df,PIPELINE_NAME,RUN_ID)
display(customers_df.limit(5))

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,ingestion_timestamp,ingestion_date,pipeline_name,run_id
CUST_000001,UNIQ_000001,13278,salvador,BA,2026-07-15T07:38:01.139Z,2026-07-15,Bronze_Customers,c5cd7a7e-f6cb-4a1a-b298-b80c72a8ca3b
CUST_000002,UNIQ_000002,42098,porto alegre,RS,2026-07-15T07:38:01.139Z,2026-07-15,Bronze_Customers,c5cd7a7e-f6cb-4a1a-b298-b80c72a8ca3b
CUST_000003,UNIQ_000003,28289,recife,PE,2026-07-15T07:38:01.139Z,2026-07-15,Bronze_Customers,c5cd7a7e-f6cb-4a1a-b298-b80c72a8ca3b
CUST_000004,UNIQ_000004,98696,salvador,BA,2026-07-15T07:38:01.139Z,2026-07-15,Bronze_Customers,c5cd7a7e-f6cb-4a1a-b298-b80c72a8ca3b
CUST_000005,UNIQ_000005,21395,maceio,AL,2026-07-15T07:38:01.139Z,2026-07-15,Bronze_Customers,c5cd7a7e-f6cb-4a1a-b298-b80c72a8ca3b


In [0]:
status, error = write_bronze_table(customers_df,TARGET_TABLE)

Bronze table written: retailmart.bronze.customers


In [0]:
bronze_df = spark.table(TARGET_TABLE)
rows_written = bronze_df.count()
print(f"Rows Written : {rows_written}")
display(bronze_df.limit(5))

Rows Written : 15000


customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,ingestion_timestamp,ingestion_date,pipeline_name,run_id
CUST_000001,UNIQ_000001,13278,salvador,BA,2026-07-15T07:38:12.120Z,2026-07-15,Bronze_Customers,c5cd7a7e-f6cb-4a1a-b298-b80c72a8ca3b
CUST_000002,UNIQ_000002,42098,porto alegre,RS,2026-07-15T07:38:12.120Z,2026-07-15,Bronze_Customers,c5cd7a7e-f6cb-4a1a-b298-b80c72a8ca3b
CUST_000003,UNIQ_000003,28289,recife,PE,2026-07-15T07:38:12.120Z,2026-07-15,Bronze_Customers,c5cd7a7e-f6cb-4a1a-b298-b80c72a8ca3b
CUST_000004,UNIQ_000004,98696,salvador,BA,2026-07-15T07:38:12.120Z,2026-07-15,Bronze_Customers,c5cd7a7e-f6cb-4a1a-b298-b80c72a8ca3b
CUST_000005,UNIQ_000005,21395,maceio,AL,2026-07-15T07:38:12.120Z,2026-07-15,Bronze_Customers,c5cd7a7e-f6cb-4a1a-b298-b80c72a8ca3b


end_time = end_pipeline()
duration = execution_time(START_TIME, end_time)

print("BRONZE LOAD REPORT")
print(f"Source File        : {SOURCE_FILE}")
print(f"Target Table       : {TARGET_TABLE}")
print(f"Run ID             : {RUN_ID}")
print(f"Start Time         : {START_TIME}")
print(f"End Time           : {end_time}")
print(f"Duration           : {duration} sec")
print(f"Rows Read          : {total_rows}")
print(f"Rows Written       : {rows_written}")
print(f"Duplicate Rows     : {duplicate_rows}")
print(f"PK Validation      : {'PASSED' if pk_status else 'FAILED'}")
print(f"Schema Validation  : {schema_status}")
print(f"Status             : {status}")
print(f"Error              : {error}")

In [0]:
bronze_load_report(
    pipeline_name=PIPELINE_NAME,
    run_id=RUN_ID,
    source=SOURCE_FILE,
    target=TARGET_TABLE,
    rows_read=total_rows,
    rows_written=rows_written,
    duplicate_count=duplicate_rows,
    start_time=START_TIME,
    status=status
)

BRONZE LOAD REPORT
Pipeline        : Bronze_Customers
Run ID          : c5cd7a7e-f6cb-4a1a-b298-b80c72a8ca3b
Source          : /Volumes/dbacademy/default/raw/raw_customers_dataset.csv
Target          : retailmart.bronze.customers
Rows Read       : 15000
Rows Written    : 15000
Duplicate Rows  : 0
Start Time      : 2026-07-15 07:36:59.737479
End Time        : 2026-07-15 07:38:23.422206
Duration (sec)  : 83.68
Status          : SUCCESS
